# Statistical Approaches
This notebook contains all code that was used for various statistical approaches, including K-Means clustering, Beta and Negative Binomial Regression, and $\chi^2$ Test of Independence

---

## Download & Import Dependencies

In [ ]:
!pip install bambi
!pip install prince
!pip install dill

import dill
import random
import prince
import pandas as pd
import numpy as np
import bambi as bmb
import arviz as az
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')
import matplotlib.pyplot as plt
from scipy.special import logit, expit
from scipy.stats import chi2_contingency
from sklearn.metrics import jaccard_score
from sklearn.cluster import MiniBatchKMeans
from scipy.cluster.hierarchy import dendrogram, linkage


# Set paramenters for all plots to match thesis text
plt.rcParams.update({
    # Matches \allsectionsfont{\sffamily} and \captionsetup{font=small,labelfont=sf}
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "Liberation Sans"],

    # Matches 12pt report document; 'small' in 12pt is roughly 10-11pt
    "font.size": 10.5,
    "axes.labelsize": 11,
    "axes.titlesize": 12,

    # Clean style for academic figures
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.5,      # Matches \headrulewidth 0.5pt
    "lines.linewidth": 1.0,
})

---
## Import, Subset, and Scale Data for Analysis
This is the last preprocessing step to allow for easier analysis.

In [ ]:
df_all = pd.read_parquet("/content/drive/MyDrive/Bosnian Parliment/sentiment_analysis_results.parquet")
df_real = pd.read_parquet("/content/drive/MyDrive/Bosnian Parliment/sentiment_analysis_results_with_topics.parquet")

# Drop all classes that denote macro level codes for subsetting in previous pre-processing
macro_classes = ["SUBSTANTIVE_TECHNICAL", "PROCEDURAL", "SUBSTANTIVE_IDENTITY", "SUBSTANTIVE_TECHNICAL_bool", "PROCEDURAL_bool", "SUBSTANTIVE_IDENTITY_bool"]
df_real = df_real.drop(columns=macro_classes)

In [ ]:
s_max = df_real["sentiment_results"].max()
s_min = df_real["sentiment_results"].min()

#Scale sentiment scores to a 0-1 scale
df_real['sentiment_scaled'] = (df_real['sentiment_results'] - s_min) / (s_max - s_min)

print(df_real['sentiment_scaled'].describe())

In [ ]:
party_segements = df_real.groupby(["party"])[['segment_id']].count().sort_values('segment_id', ascending=False).reset_index()
N_2k_parties = party_segements[party_segements["segment_id"] >= 2000] # Filter parties over 2k segments
N_2k_parties = N_2k_parties['party'].to_list()
N_2k_parties
df_real_N2k = df_real[df_real["party"].isin(N_2k_parties)]


---
## Visualize Class Overlap & Totals

In [ ]:
# Create list from the list in the config file of the model
class_names_clean = {
    "0": "COLLECT_MEM",
    "1": "COND_CORRUPT",
    "2": "EXIST_IDENTITY",
    "3": "HIST_CONT",
    "4": "MORAL_AGENCY",
    "5": "OTHER_ABSTRACT",
    "6": "OTHER_EXTERNAL",
    "7": "OTHER_INTERNAL",
    "9": "ROUTINE_DEF",
    "10": "SELF_ETHNIC",
    "11": "SELF_GOV",
    "12": "SELF_PARTY",
    "13": "SELF_STATE",
    "14": "SELF_TRANS",
  }
class_names_clean = list(class_names_clean.values())

In [ ]:
# Create table of toatl counts and frequency
topic_counts = df_all[[f"{col}_bool" for col in clean_names]].sum()
topic_counts = topic_counts.sort_values(ascending=False)
topic_counts = pd.DataFrame(topic_counts)
topic_counts = topic_counts.reset_index()
topic_counts = topic_counts.rename(columns={topic_counts.columns[0]: "label", topic_counts.columns[1]: "count"})
topic_counts['percent'] = topic_counts['count'] / df_all['segment'].size * 100
topic_counts = topic_counts.round(2)
topic_counts.to_latex(index=False, escape=True)

In [ ]:
binary_matrix = df_real_N2k[[f"{col}_bool" for col in class_names_clean]].astype(int)
binary_matrix.columns = class_names_clean # Clean up names for the matrix

# Dot product creates the overlap counts
co_occurrence = binary_matrix.T.dot(binary_matrix)
def jaccard_normalize(matrix):
    counts = np.diag(matrix)
    normalized = matrix.copy().astype(float)
    for i in range(len(matrix)):
        for j in range(len(matrix)):
            union = counts[i] + counts[j] - matrix.iloc[i, j]
            if union > 0:
                normalized.iloc[i, j] = matrix.iloc[i, j] / union
            else:
                normalized.iloc[i, j] = 0
    return normalized

jaccard_matrix = jaccard_normalize(co_occurrence) # Use jaccard index for overlap
mask = np.triu(np.ones_like(jaccard_matrix, dtype=bool))

# Create plot
plt.figure(figsize=(10, 8))
sns.heatmap(
    jaccard_matrix,
    mask=mask,
    annot=True,          # Show the percentages
    fmt=".0%",           # Format as percentage
    cmap="YlGnBu",       # Scientific-style color palette
    cbar_kws={'label': 'Jaccard Similarity Index'}
)
plt.tight_layout()
plt.grid(None)
plt.savefig("class_overlap_heatmap.pdf")
plt.show()

---
## Visualize Overlap Top 10 of Topics by Party

In [ ]:
# Create the top 10 topics per party
party_top_topics = {}
for party in df_real_N2k['party'].unique():
    top_10 = df_real_N2k[df_real_N2k['party'] == party]['topic_label'].value_counts().head(10).index.tolist()
    party_top_topics[party] = set(top_10)

# Create Jaccard Matrix
matrix = []
for p1 in N_2k_parties:
    row = []
    for p2 in N_2k_parties:
        # Calculate Jaccard: Intersection / Union
        intersection = len(party_top_topics[p1] & party_top_topics[p2])
        union = len(party_top_topics[p1] | party_top_topics[p2])
        jaccard = intersection / union if union > 0 else 0
        row.append(jaccard)
    matrix.append(row)

df_jaccard = pd.DataFrame(matrix, index=N_2k_parties, columns=N_2k_parties)
mask = np.triu(np.ones_like(df_jaccard, dtype=float)) # Mask mirrored side

# Create the Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    df_jaccard,
    mask=mask,
    annot=True,          # Show the percentages
    fmt=".2%",           # Format as percentage
    cmap="YlGnBu",       # Scientific-style color palette
    cbar_kws={'label': 'Jaccard Similarity Index'}
)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("topic_overlap_heatmap.pdf")
plt.show()

---
## Class Clustering with k-Means Clustering

In [ ]:
# Check if there are any biases in the outliers (-1 Topic)
outlier_bias = df_real.groupby('party')['topic_id'].apply(lambda x: (x == -1).mean()).sort_values(ascending=False)

print("Percentage of 'Noise' (Outliers) by Party:")
print(outlier_bias)

In [ ]:
class_bool_columns = [col for col in df_real_N2k.columns if col.endswith('_bool')]

# Create embeddings for clustering
mca = prince.MCA(
    n_components=5,
    n_iter=3,
    copy=True,
    check_input=True,
    engine='sklearn',
    random_state=42
)
mca_results = mca.fit_transform(df_real_N2k[class_bool_columns])

# K-Means Clustering
cluster = MiniBatchKMeans(n_clusters=5, random_state=42)
df_real_N2k['Rhetorical_Cluster'] = cluster.fit_predict(mca_results)

# Profile the Clusters
# Check which identity markers define each cluster (Used for Appendix)
cluster_profile = df_real_N2k.groupby('Rhetorical_Cluster')[class_bool_columns].mean()
print(cluster_profile)

cluster_map = {
    0: "Institutional Othering",
    1: "Existential Statehood",
    2: "Internal Governance",
    3: "Historical/Memory Statehood",
    4: "Civic/Moral Statehood"
}
df_real_N2k['Rhetorical_Archetype'] = df_real_N2k['Rhetorical_Cluster'].map(cluster_map)
# Save final dataset used for analysis
df_real_N2k.to_parquet("/content/drive/MyDrive/Bosnian Parliment/bosnian_parliament_final_clustered.parquet", compression='zstd')

---
## $\chi^2$ Test of Independence (Archetypes & Parties)

In [ ]:
# Create the contingency table
contingency_table = pd.crosstab(df_real_N2k['party'], df_real_N2k['Rhetorical_Archetype'])

# Run the Chi-Squared Test
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Calculate Cramer's V (Effect Size)
n = contingency_table.sum().sum()
min_dim = min(contingency_table.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim))

print(f"Chi-Squared Statistic: {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.4f}")

# Visualize the Pearson Residuals
residuals = (contingency_table - expected) / np.sqrt(expected)
print("\nPearson Residuals (High positive values = Strong Association):")
print(residuals)

---
## Bayesian Negative Binomial Regression
### Topic-Archetype Interaction

In [ ]:
# Prep data (remove -1 topic)
Bayes_df = df_real_N2k[df_real_N2k['topic_id'] != -1]
# Create value counts of each interaction
Bayes_df = Bayes_df[['topic_label', 'Rhetorical_Archetype']].value_counts().reset_index()
Bayes_df.columns = ['topic_label', 'Rhetorical_Archetype', 'counts']

# Establish priors
my_priors = {
    "common": bmb.Prior("Normal", mu=0, sigma=1)
}

# Establish model (Negative Binomial Regression)
model = bmb.Model(
    "counts ~ C(topic_label) * C(Rhetorical_Archetype)",
    Bayes_df,
    family="negativebinomial",
    priors=my_priors
)

# Fit the model using MCMC (Markov Chain Monte Carlo)
# This 'samples' the data to find the most likely coefficients
results = model.fit(draws=4000, tune=2000, target_accept=0.95, init="adapt_diag")


# Return Summary
summary_df = az.summary(results).reset_index()
summary_df.rename(columns={'index': 'Parameter'}, inplace=True)

# 2. Filter for Significance (Where both the 3% and 97% bounds are on the same side of zero)

significant_results_archetype = summary_df[
    ~((summary_df['hdi_3%'] < 0) & (summary_df['hdi_97%'] > 0))
].copy()

# Save to CSV (precaution)
significant_results_archetype.to_csv("significant_bayesian_results_archetype.csv", index=False)

In [ ]:
# Diagnostic Plot
az.plot_trace(results, var_names=["C(Rhetorical_Archetype)"], kind="rank_bars")
plt.tight_layout()
plt.show()

### Topic-Party Interaction

In [ ]:
# Prep data (remove -1 topic)
Bayes_df = df_real_N2k[df_real_N2k['topic_id'] != -1]
# Create value counts of each interaction
Bayes_df = Bayes_df[['topic_label', 'party']].value_counts().reset_index()
Bayes_df.columns = ['topic_label', 'party', 'counts']

# Establish priors
my_priors = {
    "common": bmb.Prior("Normal", mu=0, sigma=1)
}

# Establish model (Negative Binomial Regression)
model = bmb.Model(
    "counts ~ C(topic_label) * C(party)",
    Bayes_df,
    family="negativebinomial",
    priors=my_priors
)

# Fit the model using MCMC (Markov Chain Monte Carlo)
# This 'samples' the data to find the most likely coefficients
results = model.fit(draws=4000, tune=2000, target_accept=0.95, init="adapt_diag")


# Return Summary
summary_df = az.summary(results).reset_index()
summary_df.rename(columns={'index': 'Parameter'}, inplace=True)

# Filter for Significance (Where both the 3% and 97% bounds are on the same side of zero)
significant_results_party = summary_df[
    ~((summary_df['hdi_3%'] < 0) & (summary_df['hdi_97%'] > 0))
].copy()


# 4. Save to CSV
significant_results_party.to_csv("significant_bayesian_results_party.csv", index=False)

In [ ]:
# Diagnostic Plot
az.plot_trace(results, var_names=["C(party)"], kind="rank_bars")
plt.tight_layout()
plt.show()

### Create Forest Plot

In [ ]:
# Archetype-Topic Interactions
# Extract interactions
interactions_arch = significant_results_archetype[significant_results_archetype['Parameter'].str.contains(':')].copy()

# Clean up the labels
# interactions['Clean_Label'] = interactions['Parameter'].str.extract(r'\[(.*)\]')
interactions_arch['Clean_Label'] = [
    "Communism/Totalitarianism :\nHistorical Statehood",
    "Political Parties :\nInstitutional Othering",
    "Fascism : Historical Statehood",
    "Genocide/Srebrenica :\nHistorical Statehood"
]

# Party-Topic Interactions
# Extract interactions
interactions_party = significant_results_party[significant_results_party['Parameter'].str.contains(':')].copy()

# Clean up the labels
# interactions['Clean_Label'] = interactions['Parameter'].str.extract(r'\[(.*)\]')
interactions_party['Clean_Label'] = [
    "Investment : none",
    "Gender Equality : SBB",
    "RS/SNSD : SDS",
    "Ministerial Appointments : none",
    "Border Crossings : none",
    "Public Television : Koalicija"
]

In [ ]:
fig, axes = plt.subplots(nrows=2, sharex=True)

axes[0].set(title='Archetype-Topic')
axes[0].errorbar(
    x=interactions_arch['mean'],
    y=interactions_arch['Clean_Label'],
    xerr=[interactions_arch['mean'] - interactions_arch['hdi_3%'],
          interactions_arch['hdi_97%'] - interactions_arch['mean']],
    fmt='o', color='black', ecolor='gray', elinewidth=2, capsize=5, label='94% HDI'
)
axes[0].axvline(0, color='red', linestyle='--', alpha=0.6)
axes[0].grid(axis='x', linestyle=':', alpha=0.5)
plt.tight_layout()


axes[1].set(title='Party-Topic', xlabel='Effect Size (Log-Count Difference)')
axes[1].errorbar(
    x=interactions_party['mean'],
    y=interactions_party['Clean_Label'],
    xerr=[interactions_party['mean'] - interactions_party['hdi_3%'],
          interactions_party['hdi_97%'] - interactions_party['mean']],
    fmt='o', color='black', ecolor='gray', elinewidth=2, capsize=5, label='94% HDI'
)
axes[1].axvline(0, color='red', linestyle='--', alpha=0.6)
axes[1].grid(axis='x', linestyle=':', alpha=0.5)
plt.tight_layout()

plt.grid(axis='x', linestyle=':', alpha=0.5)
fig.supylabel("Interactions", fontsize=12)

plt.savefig("interaction_spikes_forest_plot.pdf")
plt.show()

---
## Party Interactions (Beta Regression)



In [ ]:
# Prep Data (Remove Outlier Topics)
regression_df = df_real_N2k[df_real_N2k['topic_id'] != -1]

n = len(regression_df)
# Nudge values away from 0 and 1 (Squeeze to prevent zero/one inflation)
regression_df["sentiment_scaled"] = (regression_df["sentiment_scaled"] * (n - 1) + 0.5) / n

In [ ]:
# Find the mean intercept for mu
mu_intercept = logit(regression_df["sentiment_scaled"].mean())

In [ ]:
# Set priors
simple_priors = {
    # Center the intercept at mean
    "Intercept": bmb.Prior("Normal", mu=mu_intercept, sigma=0.2),

    # Common effect for Party (logit scale)
    # sigma=0.5 allows for large partisan swings
    "C(party)": bmb.Prior("Normal", mu=0, sigma=0.5),

    # Precision parameter (Kappa)
    "kappa": bmb.Prior("Gamma", alpha=50, beta=1),

    # Random effect for speakers (Fullname)
    "1|fullname_sigma": bmb.Prior("HalfNormal", sigma=0.2)
}

# Establish model
model_party = bmb.Model(
    "sentiment_scaled ~ C(party) + (1|fullname)",
    regression_df,
    family="beta",
    priors=simple_priors
)

# Fit model
results_party = model_party.fit(
    target_accept=0.95,
    init="adapt_diag",
    draws=1000,
    tune=1000,
    random_seed=42
)

### Diagnostics & Results

In [ ]:
inference_data = az.summary(results_party)
# Plot the 'fuzzy caterpillars' for the main effects
az.plot_trace(results_party, var_names=["Intercept", "C(party)"])

In [ ]:
# This generates the 'fit' comparison
model_party.predict(results_party, kind="pps")
az.plot_ppc(results_party)

In [ ]:
# Results
results_summary_party = az.summary(results_party, hdi_prob=0.95, var_names=["C(party)"])
results_summary_party.index = results_summary_party.index.str.replace("C(party)[", "", regex=False).str.replace("]", "", regex=False)
results_summary_party.to_latex()

In [ ]:
# Convert logit -> sentiment

intercept_logit = results_summary_party.loc["Intercept", "mean"]
print(f"Baseline Logit: {intercept_logit}")
expit(intercept_logit)

readable_summary = results_summary_party.copy()

readable_summary['mean_sentiment'] = expit(intercept_logit + readable_summary['mean'])
readable_summary['hdi_2.5%_sentiment'] = expit(intercept_logit + readable_summary['hdi_2.5%'])
readable_summary['hdi_97.5%_sentiment'] = expit(intercept_logit + readable_summary['hdi_97.5%'])

# Display the converted results
print(readable_summary[['mean_sentiment', 'hdi_2.5%_sentiment', 'hdi_97.5%_sentiment']])

---
## Topic & Archetype Effects on Sentiment (Beta Regression)
### Prep Data

In [ ]:
# Remove Outlier Topics
regression_df = df_real_N2k[df_real_N2k['topic_id'] != -1]

n = len(regression_df)
# Nudge values away from 0 and 1 (Squeeze)
regression_df["sentiment_scaled"] = (regression_df["sentiment_scaled"] * (n - 1) + 0.5) / n

In [ ]:
# Descriptive Statistics
print(regression_df['fullname'].isna().sum())
print(np.isinf(regression_df["sentiment_scaled"]).any())
print(regression_df['sentiment_scaled'].describe())

In [ ]:
# Mean Intercept
mu_intercept = logit(regression_df["sentiment_scaled"].mean())

### Model

In [ ]:
# Set priors
simple_priors = {
    # Center the intercept at mean
    "Intercept": bmb.Prior("Normal", mu=mu_intercept, sigma=0.2),

    # Common effect for Party (logit scale)
    # sigma=0.5 is quite wide for logit; it allows for large partisan swings
    "C(party)": bmb.Prior("Normal", mu=0, sigma=0.5),

    # Precision parameter (Kappa)
    "kappa": bmb.Prior("Gamma", alpha=50, beta=1),

    # Random effect for speakers (Fullname)
    "1|fullname_sigma": bmb.Prior("HalfNormal", sigma=0.2)
}

model = bmb.Model(
    "sentiment_scaled ~ C(Rhetorical_Archetype) + C(topic_label) + (1|fullname)",
    regression_df,
    family="beta",
    priors=simple_priors
)

# Use VI due to computational issues
results_vi = model.fit(
    method="vi",
    n=40000,
    draws=1000,
    random_seed=42
)


### Diagnostics

In [ ]:
plt.plot(results_vi.hist)
plt.title("ELBO Convergence (Should flatten out)")
plt.xlabel("Iteration")
plt.ylabel("ELBO")
plt.show()

In [ ]:
inference_data = results_vi.sample(1000)
# Plot the 'fuzzy caterpillars' for the main effects
az.plot_trace(inference_data, var_names=["Intercept", "C(topic_label)", "C(Rhetorical_Archetype)"])

In [ ]:
# This generates the 'fit' comparison
model.predict(inference_data, kind="pps")
az.plot_ppc(inference_data)

In [ ]:
results_summary = az.summary(inference_data, hdi_prob=0.95, var_names=["Intercept", "C(topic_label)", "C(Rhetorical_Archetype)"])
print(results_summary)

# To save it to CSV for your paper
results_summary.to_csv("double_bayesian_results_summary.csv")

In [ ]:
significant_results = results_summary[
    ~((results_summary['hdi_2.5%'] < 0) & (results_summary['hdi_97.5%'] > 0))
].copy()

### Visualize

In [ ]:
significant_topic = significant_results[significant_results.index.str.contains("C(topic_label)", regex=False)].copy()
significant_archetype = significant_results[significant_results.index.str.contains("C(Rhetorical_Archetype)", regex=False)].copy()
significant_archetype = significant_archetype.reset_index()
significant_archetype = significant_archetype.rename(columns={'index': 'Rhetorical_Archetype'})
significant_topic_top = significant_topic.sort_values(by='mean', ascending=False).head(5).copy()
significant_topic_bottom = significant_topic.sort_values(by='mean', ascending=True).head(5).copy()
significant_topic = pd.concat([significant_topic_top, significant_topic_bottom]).reset_index()
significant_topic = significant_topic.rename(columns={'index': 'topic_label'})
significant_archetype['Rhetorical_Archetype'] = significant_archetype['Rhetorical_Archetype'].str.replace("C(Rhetorical_Archetype)[", "", regex=False).str.replace("]", "", regex=False)
significant_topic.head(10)

In [ ]:
# Clean up the labels
# interactions['Clean_Label'] = interactions['Parameter'].str.extract(r'\[(.*)\]')
significant_topic['Clean_Label'] = [
    "Military Commissioner",
    "Totalitarianism/Communism",
    "Criminal Sanctions",
    "Sports",
    "Šarović/Kucić",
    "Border Crossing",
    "Nuclear Waste",
    "Islamic Terrorism",
    "War Crimes & Prosecution",
    "Corruption"
]


In [ ]:
fig, axes = plt.subplots(nrows=2, sharex=True)

axes[0].set(title='Archetype', ylabel="Archetype")
axes[0].errorbar(
    x=significant_archetype['mean'],
    y=significant_archetype['Rhetorical_Archetype'],
    xerr=[significant_archetype['mean'] - significant_archetype['hdi_2.5%'],
          significant_archetype['hdi_97.5%'] - significant_archetype['mean']],
    fmt='o', color='black', ecolor='gray', elinewidth=2, capsize=5, label='95% HDI'
)
axes[0].axvline(0, color='red', linestyle='--', alpha=0.6)
axes[0].grid(axis='x', linestyle=':', alpha=0.5)
plt.tight_layout()


axes[1].set(title='Topic', xlabel='Effect Size (Logits)', ylabel="Topic")
axes[1].errorbar(
    x=significant_topic['mean'],
    y=significant_topic['Clean_Label'],
    xerr=[significant_topic['mean'] - significant_topic['hdi_2.5%'],
          significant_topic['hdi_97.5%'] - significant_topic['mean']],
    fmt='o', color='black', ecolor='gray', elinewidth=2, capsize=5, label='95% HDI'
)
axes[1].axvline(0, color='red', linestyle='--', alpha=0.6)
axes[1].grid(axis='x', linestyle=':', alpha=0.5)
plt.tight_layout()

plt.grid(axis='x', linestyle=':', alpha=0.5)


plt.savefig("sentiment_forest_plot.pdf")
plt.show()

---
## Full Interactions (Beta Regression)
Due to computation time, Variational Inference was used instead of Monte Carlo. Interactions with less than 5 occurances were also ommitted due to computational limits
### Prepare Data

In [ ]:
# Filter Outlier Topics
regression_df = df_real_N2k[df_real_N2k['topic_id'] != -1]

n = len(regression_df)
# Nudge values away from 0 and 1 (Squeeze to avoid zero/one-inflation)
regression_df["sentiment_scaled"] = (regression_df["sentiment_scaled"] * (n - 1) + 0.5) / n

# Descriptive Statistics
print(regression_df['fullname'].isna().sum())
print(np.isinf(regression_df["sentiment_scaled"]).any())
print(regression_df['sentiment_scaled'].describe())

In [ ]:
# Create a single combined column for interaction
regression_df['topic_archetype'] = (
    regression_df['topic_label'].astype(str) + "_" +
    regression_df['Rhetorical_Archetype'].astype(str)
)

# Identify interactions with at least 5 observations
valid_levels = regression_df['topic_archetype'].value_counts()
valid_levels = valid_levels[valid_levels >= 5].index

# Filter
filtered_regression_df = regression_df[regression_df['topic_archetype'].isin(valid_levels)].copy()
print(f"Unique interactions: {filtered_regression_df['topic_archetype'].unique().size}")
print(filtered_regression_df.describe())

### Create & Fit Model

In [ ]:
# Establish priors
my_priors = {
    "Intercept": bmb.Prior("Normal", mu=-2.2, sigma=0.1),
    "common": bmb.Prior("Normal", mu=0, sigma=0.5),
    "kappa": bmb.Prior("Gamma", alpha=100, beta=1),
    "1|fullname_sigma": bmb.Prior("HalfNormal", sigma=0.1)
}

# Establish model
model = bmb.Model(
    "sentiment_scaled ~ topic_archetype * C(party) + (1|fullname)",
    filtered_regression_df,
    family="beta",
    priors=my_priors
)

# Fit model
results_vi = model.fit(
    method="vi",
    n=50000,
    draws=1000,
)

In [ ]:
# Optional (Load model from past run)
# Fitting the model can take over 2 hours

with open("/content/drive/MyDrive/Bosnian Parliment/bambi_model_full.dill", "rb") as f:
     loaded_data = dill.load(f)
     model_full = loaded_data["model"]
     inference_data = loaded_data["results"]

### Diagnostics

In [ ]:
# Plot the ELBO to check for convergence
plt.plot(results_vi.hist)
plt.title("ELBO Convergence (Should flatten out)")
plt.xlabel("Iteration")
plt.ylabel("ELBO")
plt.show()

In [ ]:
inference_data = results_vi.sample(1000)
# Plot the 'fuzzy caterpillars' for the main effects
az.plot_trace(inference_data, var_names=["Intercept", "C(party)"])

In [ ]:
# Visualize the effects of your topic-archetype combinations
az.plot_forest(inference_data, var_names=["topic_archetype"], combined=True, hdi_prob=0.95)

### Work With Results

In [ ]:
# Generate the formal results table
results_summary_full = az.summary(inference_data, hdi_prob=0.95)
print(results_summary_full)

# To save it to CSV for your paper
results_summary_full.to_csv("triple_bayesian_results_summary.csv")

In [ ]:
results_interactions = results_summary_full[
    results_summary_full.index.str.contains("C(topic_archetype):C(party)", regex=False)
    ].copy()

In [ ]:
significant_results_full = results_interactions[
    ~((results_interactions['hdi_2.5%'] < 0) & (results_interactions['hdi_97.5%'] > 0))
].copy()

In [ ]:
significant_results_latex = significant_results_full.sort_values(by='mean', ascending=False).drop(columns=["r_hat"])
significant_results_latex.to_csv("significant_interaction_results_full.csv")
